# Notebook 03 — Visualizações: OEE, Consumo e Anomalias
**Projeto:** Steel Plant Analytics
**Autor:** Ariel Marquezin
**Stack:** Matplotlib · Seaborn · pandas

In [ ]:
!pip install matplotlib seaborn pandas pyarrow -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pyarrow.parquet as pq

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#8b949e', 'axes.labelcolor': '#ffffff',
    'axes.titlecolor': '#ffffff', 'xtick.color': '#ffffff',
    'ytick.color': '#ffffff', 'text.color': '#ffffff',
    'grid.color': '#30363d', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'font.family': 'monospace', 'font.size': 12,
    'axes.titlesize': 15, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

C_CYAN   = '#00e5ff'
C_GREEN  = '#10b981'
C_YELLOW = '#f59e0b'
C_RED    = '#ef4444'
C_BLUE   = '#3b82f6'
C_PURPLE = '#7c3aed'

GOLD_PATH    = '/content/drive/MyDrive/steel/gold/'
FIGURES_PATH = '/content/drive/MyDrive/steel/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

def load(name):
    return pq.read_table(os.path.join(GOLD_PATH, f'{name}.parquet')).to_pandas()

df       = load('steel_gold_completo')
df_turno = load('metricas_turno')
df_dia   = load('metricas_diario')
anomalias = load('anomalias_detectadas')
energy_col = [c for c in df.columns if 'kwh' in c.lower() or 'consumption' in c.lower()][0]
print('Dados carregados OK')
print(f'Coluna de energia: {energy_col}')

In [ ]:
# FIGURA 1: OEE Energetico por Turno
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

oee_colors = [C_GREEN if v >= 85 else C_YELLOW if v >= 75 else C_RED for v in df_turno['oee_pct']]
bars = ax.bar(df_turno['turno'], df_turno['oee_pct'], color=oee_colors,
              width=0.5, edgecolor='#0d1117')

ax.axhline(85, color=C_GREEN, linestyle='--', linewidth=1.5, label='Meta Bom (85%)', alpha=0.8)
ax.axhline(75, color=C_YELLOW, linestyle='--', linewidth=1.5, label='Meta Aceitavel (75%)', alpha=0.8)
ax.axhline(65, color=C_RED, linestyle='--', linewidth=1.5, label='Critico (65%)', alpha=0.8)

for bar, val in zip(bars, df_turno['oee_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold', color='white')

ax.set_ylim(0, 115)
ax.set_ylabel('OEE Energetico (%)', fontsize=13)
ax.set_title('OEE Energetico por Turno — Steel Plant\n(eficiencia do consumo vs. turno de referencia)',
             fontsize=15, fontweight='bold', pad=15)
ax.legend(facecolor='#161b22', edgecolor='#8b949e', labelcolor='white')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '01_oee_turno.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 1 salva')

In [ ]:
# FIGURA 2: Consumo por Turno - Boxplot
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

turno_order = ['Turno A (06-14h)', 'Turno B (14-22h)', 'Turno C (22-06h)']
turno_order = [t for t in turno_order if t in df['turno'].unique()]

sns.boxplot(data=df, x='turno', y=energy_col, order=turno_order,
            palette=[C_CYAN, C_GREEN, C_PURPLE],
            width=0.5, fliersize=3, linewidth=1.2, ax=ax)

ax.set_xlabel('Turno', fontsize=13)
ax.set_ylabel('Consumo (kWh)', fontsize=13)
ax.set_title('Distribuicao de Consumo Energetico por Turno',
             fontsize=15, fontweight='bold', pad=15)
ax.tick_params(labelsize=11, colors='white')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '02_consumo_boxplot_turno.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 2 salva')

In [ ]:
# FIGURA 3: Tendencia Diaria com Anomalias
df_dia['data'] = pd.to_datetime(df_dia['data'])
anomalias['data_dia'] = pd.to_datetime(anomalias['timestamp']).dt.date
anomalias_dia = anomalias.groupby('data_dia').size().reset_index(name='count_anomalias')
anomalias_dia['data_dia'] = pd.to_datetime(anomalias_dia['data_dia'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                                gridspec_kw={'height_ratios': [2.5, 1]})
for a in [ax1, ax2]: a.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')

ax1.fill_between(df_dia['data'], df_dia['consumo_kwh'], alpha=0.15, color=C_CYAN)
ax1.plot(df_dia['data'], df_dia['consumo_kwh'], color=C_CYAN, linewidth=1.5, label='Consumo Diario')
ax1.plot(df_dia['data'], df_dia['media_7d'], color=C_YELLOW, linewidth=2,
         linestyle='--', label='Media 7 dias')

# Marca anomalias no grafico diario
df_dia_merged = df_dia.merge(anomalias_dia, left_on='data', right_on='data_dia', how='left')
mask_anom = df_dia_merged['count_anomalias'] > 0
ax1.scatter(df_dia_merged.loc[mask_anom, 'data'],
            df_dia_merged.loc[mask_anom, 'consumo_kwh'],
            color=C_RED, s=50, zorder=5, label='Dia com anomalia')

ax1.set_ylabel('Consumo (kWh)', fontsize=13)
ax1.set_title('Tendencia de Consumo Energetico Diario — Steel Plant',
              fontsize=15, fontweight='bold', pad=12)
ax1.legend(facecolor='#161b22', edgecolor='#8b949e', labelcolor='white')
ax1.grid(alpha=0.25)

var_colors = [C_GREEN if v >= 0 else C_RED for v in df_dia['variacao_pct'].fillna(0)]
ax2.bar(df_dia['data'], df_dia['variacao_pct'].fillna(0), color=var_colors, alpha=0.8)
ax2.axhline(0, color='#8b949e', linewidth=1)
ax2.set_ylabel('Variacao (%)', fontsize=12)
ax2.grid(axis='y', alpha=0.25)
ax2.tick_params(axis='x', labelsize=10, rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '03_tendencia_diaria.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 3 salva')

In [ ]:
# FIGURA 4: Anomalias por Turno e Hora
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for a in [ax1, ax2]: a.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')

# Por turno
anom_turno = df.groupby('turno')['anomalia'].agg(['sum','mean']).reset_index()
anom_turno.columns = ['turno','total','taxa']
anom_turno['taxa_pct'] = (anom_turno['taxa'] * 100).round(2)
colors_t = [C_RED if v > anom_turno['taxa_pct'].mean() else C_YELLOW for v in anom_turno['taxa_pct']]
bars1 = ax1.bar(anom_turno['turno'], anom_turno['taxa_pct'], color=colors_t, width=0.5, edgecolor='#0d1117')
for bar, val in zip(bars1, anom_turno['taxa_pct']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold', color='white')
ax1.set_title('Taxa de Anomalia por Turno', fontsize=14, fontweight='bold')
ax1.set_ylabel('Taxa de Anomalia (%)', fontsize=12)
ax1.tick_params(labelsize=10, colors='white')
ax1.grid(axis='y', alpha=0.25)

# Por hora do dia
anom_hora = df.groupby('hora')['anomalia'].mean().reset_index()
anom_hora['taxa_pct'] = (anom_hora['anomalia'] * 100).round(2)
colors_h = [C_RED if v > anom_hora['taxa_pct'].mean() * 1.5 else C_CYAN for v in anom_hora['taxa_pct']]
ax2.bar(anom_hora['hora'], anom_hora['taxa_pct'], color=colors_h, edgecolor='#0d1117')
ax2.set_title('Taxa de Anomalia por Hora do Dia', fontsize=14, fontweight='bold')
ax2.set_xlabel('Hora', fontsize=12)
ax2.set_ylabel('Taxa de Anomalia (%)', fontsize=12)
ax2.set_xticks(range(0, 24, 2))
ax2.tick_params(labelsize=10, colors='white')
ax2.grid(axis='y', alpha=0.25)

fig.suptitle('Mapa de Anomalias — Quando Ocorrem os Picos?',
             fontsize=15, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '04_anomalias_turno_hora.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 4 salva')

In [ ]:
# FIGURA 5: Heatmap Consumo por Hora x Dia da Semana
pivot = df.pivot_table(values=energy_col, index='hora', columns='dia_semana', aggfunc='mean')
dias  = {0:'Seg', 1:'Ter', 2:'Qua', 3:'Qui', 4:'Sex', 5:'Sab', 6:'Dom'}
pivot.columns = [dias.get(c, c) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, linecolor='#0d1117',
            ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})

# Cor das anotacoes dinamica
vmax = pivot.max().max()
vmin = pivot.min().min()
for text in ax.texts:
    val = float(text.get_text())
    text.set_color('#111111' if val < (vmax - vmin) * 0.5 + vmin else 'white')

ax.set_title('Consumo Medio (kWh) por Hora x Dia da Semana\n(identifica padroes de pico operacional)',
             fontsize=15, fontweight='bold', color='white', pad=15)
ax.set_xlabel('Dia da Semana', fontsize=13)
ax.set_ylabel('Hora do Dia', fontsize=13)
ax.tick_params(labelsize=11, colors='white')
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10, colors='white')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '05_heatmap_hora_dia.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 5 salva')

In [ ]:
# FIGURA 6: Dashboard Executivo
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
gs  = GridSpec(3, 4, figure=fig, hspace=0.6, wspace=0.4)

def kpi_box(ax, title, value, subtitle='', color=C_CYAN):
    ax.set_facecolor('#161b22')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis('off')
    ax.text(0.5, 0.72, value, ha='center', va='center',
            fontsize=18, fontweight='bold', color=color)
    ax.text(0.5, 0.38, title, ha='center', va='center',
            fontsize=10, color='white')
    if subtitle:
        ax.text(0.5, 0.16, subtitle, ha='center', va='center',
                fontsize=9, color='#8b949e')
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)

total_kwh     = df[energy_col].sum()
media_kwh     = df[energy_col].mean()
max_kwh       = df[energy_col].max()
n_anomalias   = df['anomalia'].sum()
pct_anomalias = df['anomalia'].mean() * 100
melhor_turno  = df_turno.loc[df_turno['oee_pct'].idxmax(), 'turno'].split()[1]
oee_medio     = df_turno['oee_pct'].mean()
n_dias        = len(df_dia)

kpi_box(fig.add_subplot(gs[0,0]), 'Consumo Total', f'{total_kwh/1e6:.2f}M kWh', color=C_CYAN)
kpi_box(fig.add_subplot(gs[0,1]), 'Consumo Medio/Leitura', f'{media_kwh:.1f} kWh', color=C_BLUE)
kpi_box(fig.add_subplot(gs[0,2]), 'Pico Maximo', f'{max_kwh:.1f} kWh', color=C_YELLOW)
kpi_box(fig.add_subplot(gs[0,3]), 'OEE Medio', f'{oee_medio:.1f}%',
        color=C_GREEN if oee_medio >= 75 else C_RED)

kpi_box(fig.add_subplot(gs[1,0]), 'Anomalias Detectadas', f'{n_anomalias:,}',
        f'{pct_anomalias:.2f}% dos registros', color=C_RED)
kpi_box(fig.add_subplot(gs[1,1]), 'Melhor Turno OEE', melhor_turno, color=C_GREEN)
kpi_box(fig.add_subplot(gs[1,2]), 'Dias Analisados', f'{n_dias}', color=C_CYAN)
kpi_box(fig.add_subplot(gs[1,3]), 'Total Registros', f'{len(df):,}', '(leituras de 15min)', color=C_BLUE)

ax_bar = fig.add_subplot(gs[2,:])
ax_bar.set_facecolor('#161b22')
df_turno_sorted = df_turno.sort_values('oee_pct', ascending=False)
colors_oee = [C_GREEN if v >= 85 else C_YELLOW if v >= 75 else C_RED for v in df_turno_sorted['oee_pct']]
ax_bar.barh(df_turno_sorted['turno'], df_turno_sorted['oee_pct'],
            color=colors_oee, edgecolor='#0d1117')
ax_bar.axvline(75, color=C_YELLOW, linestyle='--', linewidth=1.5, label='Meta 75%')
ax_bar.axvline(85, color=C_GREEN, linestyle='--', linewidth=1.5, label='Meta 85%')
ax_bar.set_title('OEE Energetico por Turno', fontsize=13, fontweight='bold', pad=10)
ax_bar.set_xlabel('OEE (%)', fontsize=12)
ax_bar.tick_params(labelsize=11, colors='white')
ax_bar.legend(facecolor='#161b22', edgecolor='#8b949e', labelcolor='white', fontsize=10)
ax_bar.grid(axis='x', alpha=0.25)

fig.suptitle('Dashboard Executivo — Steel Plant Analytics',
             fontsize=17, fontweight='bold', color='white', y=1.01)
plt.savefig(os.path.join(FIGURES_PATH, '06_dashboard_executivo.png'),
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Figura 6 (Dashboard) salva')

In [ ]:
print('Todas as figuras geradas:')
for f in sorted(os.listdir(FIGURES_PATH)):
    if f.endswith('.png'):
        kb = os.path.getsize(os.path.join(FIGURES_PATH, f)) / 1024
        print(f'  {f:<50} {kb:6.0f} KB')
print('Notebook 03 concluido!')